# BriskVLA evaluation on Colab

Runs `project/briskvla/run_eval.sh` for one model, the same script used on the RTX 3080, so the
settings are identical. Results are saved to Google Drive, and finished suites are skipped, so after a
disconnect you can just run all cells again.

1. *Runtime → Change runtime type →* GPU.
2. Edit the **Config** cell and run all cells.

Latency is reported on the RTX 3080, so it is off here by default.

## Config

In [ ]:
BRANCH = "vyvas/student-expert"

NAME = "teacher"      # name of the output folders
POLICY = "teacher"    # "teacher" or a checkpoint folder, e.g. "/content/students/k8_pairs"

STEPS = "10 3"
SUITES = "libero_spatial libero_object libero_goal libero_10"
EPISODES = 10         # per task
SUCCESS = 1           # 0 to skip the LIBERO success evals
LATENCY = 0           # 1 to also measure latency on this GPU

OUT = "/content/drive/MyDrive/briskvla/outputs"
REPO_URL = "https://github.com/vyvas33/briskvla.git"
REPO_DIR = "/content/briskvla"

## 1. GPU and Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

from google.colab import drive

drive.mount("/content/drive")

## 2. Install

In [ ]:
import os
import shutil

!apt-get -qq install -y libegl1 libgl1 ffmpeg > /dev/null
if shutil.which("uv") is None:
    !pip install -q uv

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} fetch -q origin && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} pull -q --ff-only
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log --oneline -1

# Colab's own Python is newer than what the repo is tested with, so pin 3.12
!cd {REPO_DIR} && uv sync --locked --inexact --python 3.12 --extra smolvla --extra libero

## 3. Optional: build an untrained student

Only needed to evaluate a student right after copying the teacher's layers. Set `POLICY` in the Config
cell to the folder below and give it its own `NAME`, e.g. `k8_pairs_untrained`.

In [ ]:
STUDENT_LAYERS = "0,1,4,5,8,9,12,13"
STUDENT_DIR = "/content/students/k8_pairs"

!cd {REPO_DIR} && uv run --no-sync python project/briskvla/build_student.py --layers {STUDENT_LAYERS} --out {STUDENT_DIR}

## 4. Run

Starts the eval in the background and returns right away. The eval keeps running if the browser tab
freezes or reloads, as long as the Colab machine is alive. Its short output goes to `run_<NAME>.log`
in the output folder on Drive, and the full output of each suite to `eval/<NAME>_K<steps>/<suite>.log`.

In [ ]:
import os
import subprocess

os.environ.update({
    "STEPS": STEPS,
    "SUITES": SUITES,
    "EPISODES": str(EPISODES),
    "SUCCESS": str(SUCCESS),
    "LATENCY": str(LATENCY),
    "OUT": OUT,
    "CKPT": "/content/checkpoints",
    "PYTHON": "3.12",
})
RUN_LOG = f"{OUT}/run_{NAME}.log"


def running():
    return subprocess.run(["pgrep", "-f", f"run_eval.sh {NAME} "], capture_output=True).returncode == 0


if running():
    print(f"{NAME} is already running, see the next cell")
else:
    os.makedirs(OUT, exist_ok=True)
    !cd {REPO_DIR} && nohup bash project/briskvla/run_eval.sh {NAME} {POLICY} >> {RUN_LOG} 2>&1 &
    print(f"Started. Log: {RUN_LOG}")

## 5. Progress

Prints a short status every 5 minutes until the eval ends. Stopping this cell does not stop the eval.
After a tab reload, run the Config cell, the cell above (it will not start a second run) and this one.

In [ ]:
import glob
import time


def status():
    print(time.strftime("%H:%M"))
    if os.path.exists(RUN_LOG):
        lines = open(RUN_LOG, errors="replace").read().splitlines()
        print("\n".join(line[:200] for line in lines[-6:]))
    for suite_dir in sorted(glob.glob(f"{OUT}/eval/{NAME}_K*/libero_*/")):
        if not os.path.exists(suite_dir + "eval_info.json"):
            tasks = len(glob.glob(suite_dir + "videos/*/"))
            print(f"in progress: {suite_dir.split('/eval/')[1]}, {tasks}/10 tasks have results")
    print()


status()
while running():
    time.sleep(300)
    status()
print("Not running")

## 6. Summary of everything in Drive

In [ ]:
!cd {REPO_DIR} && uv run --no-sync python project/briskvla/summarize_eval.py --root {OUT}/eval